**I collected every FDA safety report for 5 rehab device types from 2016 to 2026: 477 reports, validated against the FDA's own count, with event details, patient outcomes, problem codes and narratives.**

# Case Study 1 – Project Log

| Date | Step | What I found | Decision and why |
|---|---|---|---|
| 29 Sep | API test | 20 of ~500 TENS reports have a blank event type | Define "serious harm" using patient outcome codes as well, not event type alone |
| 29 Sep | Product codes | The largest code matching "transcutaneous" (LDD, 16,581 reports) is a cardiac defibrillator | Verified every code against the FDA classification database before selecting |
| 29 Sep | Product codes | Chose GZJ, NUH (TENS), IPF (muscle stimulator), PHL (exoskeleton), BXB (powered exercise incl. CPM) | Excluded cardiac, migraine and cosmetic stimulator codes as not rehab |
| 29 Sep | File check | 3 files differ from FDA documentation (2 lookups are comma-separated with a BOM; patient problem file has a header) | Adjusted readers. Always inspect raw files before processing |
| 29 Sep | Device filter | 477 unique reports (2016–Sep 2026) | Validated against openFDA API: 0.0% difference for all 5 codes |
| 29 Sep | Sample size | Only TENS has enough reports for supplier comparison | Manufacturer comparisons only for groups with 30+ reports; exoskeleton and exercise equipment get simple counts |
| 29 Sep | File check | Master file has an undocumented EVENT_KEY column | Will use it to identify duplicate reports of the same incident |
| 29 Sep | Coverage | Master 100%, patients 99.8%, device problems 93%, patient problems 100% | Failure-mode results are based on the 442 reports with device problem codes |
| 29 Sep | Duplicates | 36% of device problem rows and 48% of patient problem rows were duplicates (historical and current files overlap) | De-duplicate by report key + problem code before analysis |
| 29 Sep | Narratives | 793 texts, no duplicates, 99.6% of reports covered; no manufacturer evaluation texts (type E) | Supplier analysis will use type N narratives and the device-evaluated flag |

In [23]:
import os, subprocess
import pandas as pd

BASE = '/content/drive/MyDrive/cs1-device-safety'
FDA = 'https://www.accessdata.fda.gov/MAUDE/ftparea/'

for fname, name in [('deviceproblemcodes2025.zip', 'dev_lookup'),
                    ('patientproblemcodes2026.zip', 'pat_lookup')]:
    path = f'/content/{fname}'
    subprocess.run(['wget', '-q', '-U', 'Mozilla/5.0', FDA + fname, '-O', path], check=True)
    df = pd.read_csv(path, compression='zip', dtype=str, encoding='utf-8-sig')
    df.to_parquet(f'{BASE}/data/filtered/{name}.parquet')
    print(name, df.shape)
    print(df.head(3), '\n')

dev_lookup (885, 4)
  FDA_CODE                     TERM NCIT_CODE IMDRF_CODE
0        1    Old To Be Deactivated       NaN        NaN
1     1001  Failure To Run On AC/DC       NaN        NaN
2     1002                 Abnormal       NaN        NaN 

pat_lookup (1172, 4)
  FDA_CODE                     TERM NCIT_CODE IMDRF_CODE
0        1    Old To Be Deactivated       NaN        NaN
1     1125  Contraindicated patient       NaN        NaN
2     1156             Delamination       NaN        NaN 



In [4]:
import requests, json, csv, subprocess
import pandas as pd
from google.colab import userdata

API_KEY = userdata.get('OPENFDA_KEY')
API = 'https://api.fda.gov/device/event.json'

def count(search, field, limit=20):
    r = requests.get(API, params={'api_key': API_KEY, 'search': search, 'count': field, 'limit': limit})
    r.raise_for_status()
    return pd.DataFrame(r.json()['results'])

def api_total(code):
    r = requests.get(API, params={'api_key': API_KEY, 'limit': 1,
                                  'search': f'device.device_report_product_code:"{code}"'})
    return r.json()['meta']['results']['total']

print(count('device.device_report_product_code:"GZJ"', 'event_type.exact'))

          term  count
0       Injury    292
1        Other     96
2  Malfunction     91
3                  20
4        Death      4


In [6]:
for name in ['transcutaneous', 'exoskeleton', 'passive motion']:
    print('\n==', name)
    print(count(f'device.generic_name:"{name}"', 'device.device_report_product_code.exact', 10))


== transcutaneous
  term  count
0  LDD    947
1  GZJ    125
2  DRO     98
3  NUH     82
4  NFO     37
5  PCC     14
6  MKJ     11
7  MQM      6
8  PFB      5
9  IPF      4

== exoskeleton
  term  count
0  PHL     44

== passive motion
  term  count
0  BXB     20
1  ION      1
2  JOW      1


In [7]:
FDA = 'https://www.accessdata.fda.gov/MAUDE/ftparea/'

def download(fname):
    path = f'{RAW}/{fname}'
    if not os.path.exists(path):
        subprocess.run(['wget', '-q', '-U', 'Mozilla/5.0', FDA + fname, '-O', path], check=True)
    return path

def peek(fname, n=3):
    path = download(fname)
    out = subprocess.run(f'unzip -p {path} | head -{n}', shell=True,
                         capture_output=True, text=True, encoding='latin-1')
    print(out.stdout)

def read_filtered(fname, keys=None, codes=None, header=0, names=None, delete=True):
    path = download(fname)
    parts = []
    for chunk in pd.read_csv(path, sep='|', dtype=str, encoding='latin-1',
                             quoting=csv.QUOTE_NONE, on_bad_lines='skip',
                             header=header, names=names, chunksize=200_000,
                             compression='zip'):
        chunk.columns = [str(c).strip().upper() for c in chunk.columns]
        if codes is not None:
            col = [c for c in chunk.columns if 'PRODUCT_CODE' in c][0]
            chunk = chunk[chunk[col].str.strip().isin(codes)]
        if keys is not None:
            chunk = chunk[chunk.iloc[:, 0].str.strip().isin(keys)]
        parts.append(chunk)
    if delete:
        os.remove(path)
    print(f'{fname}: kept {sum(len(p) for p in parts):,} rows')
    return pd.concat(parts, ignore_index=True)

def save(df, name):
    df.to_parquet(f'{BASE}/data/filtered/{name}.parquet')

def done(name):
    return os.path.exists(f'{BASE}/data/filtered/{name}.parquet')

In [8]:
for f in ['device2025.zip', 'foidevproblem.zip', 'deviceproblemcodes2025.zip',
          'patientproblemcode.zip', 'patientproblemcodes2026.zip']:
    print('=====', f); peek(f)

===== device2025.zip
MDR_REPORT_KEY|DEVICE_EVENT_KEY|IMPLANT_FLAG|DATE_REMOVED_FLAG|DEVICE_SEQUENCE_NO|IMPLANT_DATE_YEAR|DATE_REMOVED_YEAR|SERVICED_BY_3RD_PARTY_FLAG|DATE_RECEIVED|BRAND_NAME|GENERIC_NAME|MANUFACTURER_D_NAME|MANUFACTURER_D_ADDRESS_1|MANUFACTURER_D_ADDRESS_2|MANUFACTURER_D_CITY|MANUFACTURER_D_STATE_CODE|MANUFACTURER_D_ZIP_CODE|MANUFACTURER_D_ZIP_CODE_EXT|MANUFACTURER_D_COUNTRY_CODE|MANUFACTURER_D_POSTAL_CODE|DEVICE_OPERATOR|EXPIRATION_DATE_OF_DEVICE|MODEL_NUMBER|CATALOG_NUMBER|LOT_NUMBER|OTHER_ID_NUMBER|DEVICE_AVAILABILITY|DATE_RETURNED_TO_MANUFACTURER|DEVICE_REPORT_PRODUCT_CODE|DEVICE_AGE_TEXT|DEVICE_EVALUATED_BY_MANUFACTUR|COMBINATION_PRODUCT_FLAG|UDI-DI|UDI-PUBLIC
21050632||||2151940||||2025/01/01|CARDIOSAVE HYBRID, TYPE G PLUG|SYSTEM, BALLOON, INTRA-AORTIC AND CONTROL|DATASCOPE CORP.|15 LAW DRIVE||FAIRFIELD|NJ|07004||US|07004|0HP||0998-00-0800-52|0998-00-0800-52|||Y||DSP|DA|Y|N|10607567108438|(01)10607567108438
21050633||||2151941||||2025/01/01|CARDIOSAVE HYBRID, TYP

In [9]:
candidates = ['LDD', 'GZJ', 'DRO', 'NUH', 'NFO', 'PCC', 'MKJ', 'IPF', 'PHL', 'BXB']
rows = []
for c in candidates:
    r = requests.get('https://api.fda.gov/device/classification.json',
                     params={'api_key': API_KEY, 'search': f'product_code:"{c}"', 'limit': 1}).json()
    info = r.get('results', [{}])[0]
    rows.append({'code': c,
                 'official_name': info.get('device_name'),
                 'specialty': info.get('medical_specialty_description'),
                 'class': info.get('device_class'),
                 'total_reports': api_total(c)})
print(pd.DataFrame(rows).to_string())

print('\n== muscle stimulator')
print(count('device.generic_name:"muscle stimulator"', 'device.device_report_product_code.exact', 10))

  code                                                official_name          specialty class  total_reports
0  LDD            Dc-Defibrillator, Low-Energy, (Including Paddles)     Cardiovascular     2          16581
1  GZJ           Stimulator, Nerve, Transcutaneous, For Pain Relief          Neurology     2            503
2  DRO   Pacemaker, Cardiac, External Transcutaneous (Non-Invasive)     Cardiovascular     2            600
3  NUH          Stimulator, Nerve, Transcutaneous, Over-The-Counter          Neurology     2            118
4  NFO    Stimulator, Transcutaneous Electrical, Aesthetic Purposes          Neurology     2             54
5  PCC  Stimulator, Nerve, Electrical, Transcutaneous, For Migraine          Neurology     2             32
6  MKJ             Automated External Defibrillators (Non-Wearable)     Cardiovascular     3         269686
7  IPF                                  Stimulator, Muscle, Powered  Physical Medicine     2            678
8  PHL                      

In [10]:
DEVICE_GROUPS = {
    'GZJ': 'TENS (prescription)',
    'NUH': 'TENS (over-the-counter)',
    'IPF': 'Powered muscle stimulator',
    'PHL': 'Powered exoskeleton',
    'BXB': 'Powered exercise equipment (incl. CPM)',
}
CODES = list(DEVICE_GROUPS)
YEARS = range(2016, 2026)

In [11]:
if not done('devices'):
    files = [f'device{y}.zip' for y in YEARS] + ['device.zip']
    devices = pd.concat([read_filtered(f, codes=CODES) for f in files], ignore_index=True)
    save(devices, 'devices')

devices = pd.read_parquet(f'{BASE}/data/filtered/devices.parquet')
KEYS = set(devices['MDR_REPORT_KEY'].str.strip())
print(f'{len(devices):,} device rows | {len(KEYS):,} unique reports')

device2016.zip: kept 64 rows
device2017.zip: kept 93 rows
device2018.zip: kept 65 rows
device2019.zip: kept 24 rows
device2020.zip: kept 34 rows
device2021.zip: kept 20 rows
device2022.zip: kept 13 rows
device2023.zip: kept 67 rows
device2024.zip: kept 32 rows
device2025.zip: kept 40 rows
device.zip: kept 28 rows
480 device rows | 477 unique reports


In [12]:
def api_total_since(code):
    r = requests.get(API, params={'api_key': API_KEY, 'limit': 1,
        'search': f'device.device_report_product_code:"{code}" AND date_received:[20160101 TO 20261231]'})
    return r.json()['meta']['results']['total']

check = []
for code, name in DEVICE_GROUPS.items():
    files_n = devices.loc[devices['DEVICE_REPORT_PRODUCT_CODE'].str.strip() == code, 'MDR_REPORT_KEY'].nunique()
    api_n = api_total_since(code)
    check.append({'code': code, 'device': name, 'our_files': files_n, 'api': api_n,
                  'difference_%': round(100 * (files_n - api_n) / api_n, 1)})
print(pd.DataFrame(check).to_string())

  code                                  device  our_files  api  difference_%
0  GZJ                     TENS (prescription)        213  213           0.0
1  NUH                 TENS (over-the-counter)         97   97           0.0
2  IPF               Powered muscle stimulator         73   73           0.0
3  PHL                     Powered exoskeleton         54   54           0.0
4  BXB  Powered exercise equipment (incl. CPM)         40   40           0.0


In [13]:
for f in ['mdrfoi.zip', 'mdrfoithru2025.zip',
          'patient.zip', 'patientthru2025.zip',
          'foidevproblem_thru2025.zip', 'patientproblemcode_thru2025.zip']:
    print('=====', f); peek(f, n=2)

===== mdrfoi.zip
MDR_REPORT_KEY|EVENT_KEY|REPORT_NUMBER|REPORT_SOURCE_CODE|MANUFACTURER_LINK_FLAG_|NUMBER_DEVICES_IN_EVENT|NUMBER_PATIENTS_IN_EVENT|DATE_RECEIVED|ADVERSE_EVENT_FLAG|PRODUCT_PROBLEM_FLAG|DATE_REPORT|DATE_OF_EVENT|REPROCESSED_AND_REUSED_FLAG|REPORTER_OCCUPATION_CODE|HEALTH_PROFESSIONAL|INITIAL_REPORT_TO_FDA|DATE_FACILITY_AWARE|REPORT_DATE|REPORT_TO_FDA|DATE_REPORT_TO_FDA|EVENT_LOCATION|DATE_REPORT_TO_MANUFACTURER|MANUFACTURER_CONTACT_T_NAME|MANUFACTURER_CONTACT_F_NAME|MANUFACTURER_CONTACT_L_NAME|MANUFACTURER_CONTACT_STREET_1|MANUFACTURER_CONTACT_STREET_2|MANUFACTURER_CONTACT_CITY|MANUFACTURER_CONTACT_STATE|MANUFACTURER_CONTACT_ZIP_CODE|MANUFACTURER_CONTACT_ZIP_EXT|MANUFACTURER_CONTACT_COUNTRY|MANUFACTURER_CONTACT_POSTAL|MANUFACTURER_CONTACT_AREA_CODE|MANUFACTURER_CONTACT_EXCHANGE|MANUFACTURER_CONTACT_PHONE_NO|MANUFACTURER_CONTACT_EXTENSION|MANUFACTURER_CONTACT_PCOUNTRY|MANUFACTURER_CONTACT_PCITY|MANUFACTURER_CONTACT_PLOCAL|MANUFACTURER_G1_NAME|MANUFACTURER_G1_STREET_1|MAN

In [14]:
if not done('master'):
    save(pd.concat([read_filtered(f, keys=KEYS) for f in ['mdrfoithru2025.zip', 'mdrfoi.zip']],
                   ignore_index=True), 'master')

mdrfoithru2025.zip: kept 449 rows
mdrfoi.zip: kept 28 rows


In [15]:
if not done('patients'):
    save(pd.concat([read_filtered(f, keys=KEYS) for f in ['patientthru2025.zip', 'patient.zip']],
                   ignore_index=True), 'patients')

patientthru2025.zip: kept 448 rows
patient.zip: kept 28 rows


In [16]:
if not done('dev_problems'):
    save(pd.concat([read_filtered(f, keys=KEYS, header=None, names=['MDR_REPORT_KEY', 'DEVICE_PROBLEM_CODE'])
                    for f in ['foidevproblem_thru2025.zip', 'foidevproblem.zip']],
                   ignore_index=True), 'dev_problems')

foidevproblem_thru2025.zip: kept 518 rows
foidevproblem.zip: kept 333 rows


In [17]:
if not done('pat_problems'):
    save(pd.concat([read_filtered(f, keys=KEYS) for f in ['patientproblemcode_thru2025.zip', 'patientproblemcode.zip']],
                   ignore_index=True), 'pat_problems')

patientproblemcode_thru2025.zip: kept 667 rows
patientproblemcode.zip: kept 721 rows


In [18]:
for n in ['master', 'patients', 'dev_problems', 'pat_problems']:
    df = pd.read_parquet(f'{BASE}/data/filtered/{n}.parquet')
    covered = set(df['MDR_REPORT_KEY'].str.strip()) & KEYS
    print(f'{n:13s} rows = {len(df):6,}   reports covered = {len(covered)} / {len(KEYS)}')

master        rows =    477   reports covered = 477 / 477
patients      rows =    476   reports covered = 476 / 477
dev_problems  rows =    851   reports covered = 442 / 477
pat_problems  rows =  1,388   reports covered = 477 / 477


In [19]:
dp = pd.read_parquet(f'{BASE}/data/filtered/dev_problems.parquet')
pp = pd.read_parquet(f'{BASE}/data/filtered/pat_problems.parquet')
for df in (dp, pp):
    for c in df.columns:
        df[c] = df[c].str.strip()

dp_unique = dp.drop_duplicates(['MDR_REPORT_KEY', 'DEVICE_PROBLEM_CODE'])
pp_unique = pp.drop_duplicates(['MDR_REPORT_KEY', 'PATIENT_SEQUENCE_NO', 'PROBLEM_CODE'])
print(f'Device problems:  {len(dp):,} rows -> {len(dp_unique):,} unique')
print(f'Patient problems: {len(pp):,} rows -> {len(pp_unique):,} unique')

Device problems:  851 rows -> 548 unique
Patient problems: 1,388 rows -> 721 unique


In [20]:
peek('foitext.zip', n=3)

MDR_REPORT_KEY|MDR_TEXT_KEY|TEXT_TYPE_CODE|PATIENT_SEQUENCE_NUMBER|DATE_REPORT|FOI_TEXT
23941978|388993707|D|0||IT WAS OBSERVED THAT DURING THE EVALUATION, THE GASTROINTESTINAL VIDEOSCOPE HAD A BURNED C-COVER. THERE WAS NO PATIENT INVOLVEMENT.
23941978|388993708|N|0||THE DEVICE WAS RETURNED TO OLYMPUS FOR INSPECTION. A ROOT CAUSE COULD NOT BE IDENTIFIED. BASED ON THE RESULTS OF THE INVESTIGATION, THE MOST PROBABLE CAUSE OF THE DAMAGES/ISSUES IS TRACED TO COMPONENT FAILURE, WHICH IS EXPECTED OR RANDOM COMPONENT FAILURE WITHOUT ANY DESIGN OR MANUFACTURING ISSUE. SHOULD ADDITIONAL RELEVANT INFORMATION BECOME AVAILABLE, A SUPPLEMENTAL REPORT WILL BE SUBMITTED. OLYMPUS WILL CONTINUE TO MONITOR FIELD PERFORMANCE FOR THIS DEVICE.



In [21]:
for f in [f'foitext{y}.zip' for y in YEARS] + ['foitext.zip']:
    name = 'text_' + f.replace('.zip', '')
    if not done(name):
        save(read_filtered(f, keys=KEYS), name)

foitext2016.zip: kept 88 rows
foitext2017.zip: kept 169 rows
foitext2018.zip: kept 119 rows
foitext2019.zip: kept 37 rows
foitext2020.zip: kept 52 rows
foitext2021.zip: kept 33 rows
foitext2022.zip: kept 22 rows
foitext2023.zip: kept 98 rows
foitext2024.zip: kept 69 rows
foitext2025.zip: kept 71 rows
foitext.zip: kept 35 rows


In [22]:
import glob
text = pd.concat([pd.read_parquet(p) for p in sorted(glob.glob(f'{BASE}/data/filtered/text_*.parquet'))],
                 ignore_index=True)
for c in text.columns:
    text[c] = text[c].str.strip()

before = len(text)
text = text.drop_duplicates(['MDR_REPORT_KEY', 'MDR_TEXT_KEY'])
save(text, 'narratives')

covered = set(text['MDR_REPORT_KEY']) & KEYS
print(f'{before} rows -> {len(text)} unique | reports with a narrative: {len(covered)} / {len(KEYS)}')
print(text['TEXT_TYPE_CODE'].value_counts())

793 rows -> 793 unique | reports with a narrative: 475 / 477
TEXT_TYPE_CODE
D    562
N    231
Name: count, dtype: int64
